In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score, log_loss, accuracy_score

from common.load_data import load_data, drop_targets, get_test_val_sets
from src.common.treat_missing import TabularPreprocessor, PreprocessConfig, load_and_process
from src.common.evaluations import evaluate_classification
from src.utils.plots import plot_loss_curve

import warnings

In [5]:
df_full = pd.read_csv("data/cibil_score/cibil_score.csv")
df_full = df_full.drop(columns=["Unnamed: 0"])
df_full.columns = [col.lower().strip() for col in df_full.columns]

print(df_full.shape)
df_full = df_full.drop_duplicates().reset_index(drop=True)
print("After dedup:", df_full.shape)
df_full.head()

(51336, 87)
After dedup: (51336, 87)


,prospectid,total_tl,tot_closed_tl,tot_active_tl,total_tl_opened_l6m,tot_tl_closed_l6m,pct_tl_open_l6m,pct_tl_closed_l6m,pct_active_tl,pct_closed_tl,...,pct_cc_enq_l6m_of_l12m,pct_pl_enq_l6m_of_ever,pct_cc_enq_l6m_of_ever,max_unsec_exposure_inpct,hl_flag,gl_flag,last_prod_enq2,first_prod_enq2,credit_score,approved_flag
0,1,5,4,1,0,0,0.000,0.0,0.200,0.800,...,0.0,0.0,0.0,13.333,1,0,PL,PL,696,P2
1,2,1,0,1,0,0,0.000,0.0,1.000,0.000,...,0.0,0.0,0.0,0.860,0,0,ConsumerLoan,ConsumerLoan,685,P2
2,3,8,0,8,1,0,0.125,0.0,1.000,0.000,...,0.0,0.0,0.0,5741.667,1,0,ConsumerLoan,others,693,P2
3,4,1,0,1,1,0,1.000,0.0,1.000,0.000,...,0.0,0.0,0.0,9.900,0,0,others,others,673,P2
4,5,3,2,1,0,0,0.000,0.0,0.333,0.667,...,0.0,0.0,0.0,-99999.000,0,0,AL,AL,753,P1


In [6]:
df_full.columns

Index(['prospectid', 'total_tl', 'tot_closed_tl', 'tot_active_tl',
       'total_tl_opened_l6m', 'tot_tl_closed_l6m', 'pct_tl_open_l6m',
       'pct_tl_closed_l6m', 'pct_active_tl', 'pct_closed_tl',
       'total_tl_opened_l12m', 'tot_tl_closed_l12m', 'pct_tl_open_l12m',
       'pct_tl_closed_l12m', 'tot_missed_pmnt', 'auto_tl', 'cc_tl',
       'consumer_tl', 'gold_tl', 'home_tl', 'pl_tl', 'secured_tl',
       'unsecured_tl', 'other_tl', 'age_oldest_tl', 'age_newest_tl',
       'time_since_recent_payment', 'time_since_first_deliquency',
       'time_since_recent_deliquency', 'num_times_delinquent',
       'max_delinquency_level', 'max_recent_level_of_deliq', 'num_deliq_6mts',
       'num_deliq_12mts', 'num_deliq_6_12mts', 'max_deliq_6mts',
       'max_deliq_12mts', 'num_times_30p_dpd', 'num_times_60p_dpd', 'num_std',
       'num_std_6mts', 'num_std_12mts', 'num_sub', 'num_sub_6mts',
       'num_sub_12mts', 'num_dbt', 'num_dbt_6mts', 'num_dbt_12mts', 'num_lss',
       'num_lss_6mts', 'n

In [7]:
df = df_full[['age', 'netmonthlyincome']]

## PCA Execution

In [9]:

# -------------------------
# Step 1: Standardize Data
# -------------------------
scaler = StandardScaler()

X_std = scaler.fit_transform(df)


X_std = pd.DataFrame(
    X_std,
    columns=df.columns
)

print("\nStandardized Data")
print(X_std.round(3))

# -------------------------
# Step 2: Correlation Matrix
# -------------------------
corr = X_std.corr()

print("\nCorrelation Matrix")
print(corr.round(3))

# -------------------------
# Step 3: Covariance Matrix
# -------------------------
cov_matrix = np.cov(X_std.T)

print("\nCovariance Matrix")
print(np.round(cov_matrix,3))

# -------------------------
# Step 4: Eigen Values & Eigen Vectors
# -------------------------
eigen_values, eigen_vectors = np.linalg.eig(cov_matrix)

print("\nEigen Values")
print(np.round(eigen_values,3))

print("\nEigen Vectors")
print(np.round(eigen_vectors,3))

# -------------------------
# Step 5: Sort Eigen Values
# -------------------------
idx = np.argsort(eigen_values)[::-1]

eigen_values = eigen_values[idx]
eigen_vectors = eigen_vectors[:,idx]

print("\nSorted Eigen Values")
print(np.round(eigen_values,3))

print("\nSorted Eigen Vectors")
print(np.round(eigen_vectors,3))

# -------------------------
# Step 6: Compute PC1 & PC2
# -------------------------
principal_components = np.dot(X_std, eigen_vectors)

pc_df = pd.DataFrame(
    principal_components,
    columns=["PC1", "PC2"]
)

print("\nPrincipal Components")
print(pc_df.round(3))

# -------------------------
# Step 7: Explained Variance
# -------------------------
explained_variance = eigen_values / np.sum(eigen_values)

print("\nExplained Variance Ratio")
print(np.round(explained_variance,3))


Standardized Data
         age  netmonthlyincome
0      1.615             1.227
1     -1.220            -0.371
2      0.708            -1.319
3      0.027            -0.820
4      1.615            -0.570
...      ...               ...
51331  1.048            -0.396
51332 -0.313            -0.071
51333 -0.653            -0.421
51334 -0.993            -0.680
51335 -0.993            -0.521

[51336 rows x 2 columns]

Correlation Matrix
                    age  netmonthlyincome
age               1.000             0.084
netmonthlyincome  0.084             1.000

Covariance Matrix
[[1.    0.084]
 [0.084 1.   ]]

Eigen Values
[1.084 0.916]

Eigen Vectors
[[ 0.707 -0.707]
 [ 0.707  0.707]]

Sorted Eigen Values
[1.084 0.916]

Sorted Eigen Vectors
[[ 0.707 -0.707]
 [ 0.707  0.707]]

Principal Components
         PC1    PC2
0      2.010 -0.275
1     -1.125  0.601
2     -0.432 -1.433
3     -0.561 -0.599
4      0.739 -1.546
...      ...    ...
51331  0.461 -1.021
51332 -0.272  0.171
51333 -0.759  0

In [7]:
df = df_full.copy()

In [8]:
X = df.drop(columns=["approved_flag", "credit_score"])
y_multiclass = df["approved_flag"].astype(str)
y_credit_score = df["credit_score"].astype(float)

binary_map = {"P1": 1, "P2": 1, "P3": 0, "P4": 0}
y_binary = df["approved_flag"].map(binary_map)

X_train, X_test, idx_train, idx_test = train_test_split(
    X, X.index, test_size=0.2, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)

(41068, 85) (10268, 85)


In [9]:
# Drop very-high-missingness columns
high_missing = [c for c in ["cc_utilization", "pl_utilization"] if c in X_train.columns]
X_train = X_train.drop(columns=high_missing)
X_test = X_test.drop(columns=high_missing)

# Median-impute a handful of numeric fields where -99999 means "unknown"
median_columns = ["age_oldest_tl", "age_newest_tl", "pct_currentbal_all_tl", "time_since_recent_payment"]
for col in median_columns:
    if col not in X_train.columns:
        continue
    X_train[col] = X_train[col].replace(-99999, np.nan)
    X_test[col] = X_test[col].replace(-99999, np.nan)
    train_median = X_train[col].median()
    X_train[col] = X_train[col].fillna(train_median)
    X_test[col] = X_test[col].fillna(train_median)

# Delinquency fields: -99999 means "never delinquent" -> 0
delinquency_columns = [c for c in [
    "max_delinquency_level", "max_deliq_6mts", "max_deliq_12mts",
    "time_since_recent_deliquency", "time_since_first_deliquency",
] if c in X_train.columns]
X_train[delinquency_columns] = X_train[delinquency_columns].replace(-99999, 0)
X_test[delinquency_columns] = X_test[delinquency_columns].replace(-99999, 0)

# Enquiry fields: -99999 means "no enquiry" -> 0
enquiry_columns = [c for c in [
    "tot_enq", "cc_enq", "pl_enq", "cc_enq_l6m", "cc_enq_l12m",
    "pl_enq_l6m", "pl_enq_l12m", "enq_l3m", "enq_l6m", "enq_l12m",
] if c in X_train.columns]
X_train[enquiry_columns] = X_train[enquiry_columns].replace(-99999, 0)
X_test[enquiry_columns] = X_test[enquiry_columns].replace(-99999, 0)

# time_since_recent_enq: -99999 = "no enquiry ever"
col = "time_since_recent_enq"
if col in X_train.columns:
    train_mask = X_train[col] == -99999
    test_mask = X_test[col] == -99999
    fill_value = X_train.loc[~train_mask, col].max() + 1
    X_train["no_enquiry_flag"] = train_mask.astype(int)
    X_test["no_enquiry_flag"] = test_mask.astype(int)
    X_train[col] = X_train[col].replace(-99999, fill_value)
    X_test[col] = X_test[col].replace(-99999, fill_value)

# max_unsec_exposure_inpct: -99999 = "no unsecured loan" -> 0%
col = "max_unsec_exposure_inpct"
if col in X_train.columns:
    X_train[col] = X_train[col].replace(-99999, 0)
    X_test[col] = X_test[col].replace(-99999, 0)

# Log-transform skewed income
if "netmonthlyincome" in X_train.columns:
    X_train["netmonthlyincome_log"] = np.log1p(X_train["netmonthlyincome"].clip(lower=0))
    X_test["netmonthlyincome_log"] = np.log1p(X_test["netmonthlyincome"].clip(lower=0))
    X_train = X_train.drop(columns=["netmonthlyincome"])
    X_test = X_test.drop(columns=["netmonthlyincome"])

print("Remaining NaNs (train/test):", X_train.isna().sum().sum(), X_test.isna().sum().sum())

Remaining NaNs (train/test): 0 0
